In [1]:
import pandas as pd
import ast
import os
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def seq_frac_calcu(align_result, lenth, pla_acc):
    temp_id_list = list(align_result['pident'].value_counts().index)
    add_counts_a = align_result['qstart'].value_counts()
    minor_counts_a = align_result['qend'].value_counts()
    
    align_result = align_result[align_result['sseqid'].isin(pla_acc)]
    add_counts_p = align_result['qstart'].value_counts()
    minor_counts_p = align_result['qend'].value_counts()
    
    add_num_a = 0
    add_num_p = 0
    fra_list = []
    for j in range(lenth):
        if j+1 in add_counts_a.index:
            add_num_a += add_counts_a[j+1]
        all_count = int(add_num_a)
        if j+1 in minor_counts_a.index:
            add_num_a -= minor_counts_a[j+1]
            
        if j+1 in add_counts_p.index:
            add_num_p += add_counts_p[j+1]
        pla_count = int(add_num_p)
        if j+1 in minor_counts_p.index:
            add_num_p -= minor_counts_p[j+1]
            
        try:
            fra = pla_count/all_count
        except:
            fra = 1
        fra_list.append(fra)
    return fra_list

def pla_frac_calcu(seq_record, genus_name, pla_acc, que):
    seq_record.id = f'{seq_record.id.split('|')[0]}'
    frac_list = {'original':[], 'pident_90':[], 'pident_95':[]}
    average_value = {'contig': seq_record.id, 'size': len(seq_record)}
    os.chdir('/active-data/temp/blastn')
    temp_ncl_file = open(f'temp_nucleotide_seq_{seq_record.id}.fasta', 'w+')
    SeqIO.write(seq_record, temp_ncl_file, "fasta")
    temp_ncl_file.close()
    os.system(f'blastn -query temp_nucleotide_seq_{seq_record.id}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results_{seq_record.id}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 8')
    head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
    align_result = pd.read_csv(f'blastn_results_{seq_record.id}.txt', sep = '\t|;', engine = 'python', header = None, names = head)

    for dir_label in frac_list:
        if '90' in dir_label:
            fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 90], len(seq_record), pla_acc)
        elif '95' in dir_label:
            fra_list = seq_frac_calcu(align_result[align_result['pident'] >= 95], len(seq_record), pla_acc)
        else:
            fra_list = seq_frac_calcu(align_result, len(seq_record), pla_acc)
        tot_count = sum(fra_list)
        average_value[f'average plasmid fraction-{dir_label}'] = tot_count/len(seq_record)
    os.system(f'rm temp_nucleotide_seq_{seq_record.id}.fasta')
    os.system(f'rm blastn_results_{seq_record.id}.txt')
    que.put(pd.DataFrame([average_value]))

In [3]:
from Bio import SeqIO
from tqdm import tqdm
import multiprocessing
import subprocess

def count_fasta_seqs(file_path):
    res = subprocess.run(
        ["grep", "-c", ">", file_path],
        capture_output=True,
        text=True
    )
    return int(res.stdout.strip())

data_dir = r"/active-data/datasets/IMG_PR/Cus_PR/IMG_VR_2023-08-08_1"
plas_data = pd.read_csv(f"{data_dir}/IMGPR_plasmid_data.tsv", sep='\t')
for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    pla_acc = []
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        for item in pla_data:
            pla_acc.append(acc_n + '-' + item)
    
    target_dir = rf"/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}"
    temp_data = plas_data[plas_data['host_taxonomy'].str.contains(f'g__{genus_name}', na=False)]
    temp_data.to_csv(f'{target_dir}/IMGPR_plasmid_data.tsv', sep='\t', index=False)
    
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 10
    nucl_file = f'{target_dir}/IMGPR_nucl.fasta'
    tot = count_fasta_seqs(nucl_file)
    pool = multiprocessing.Pool(par)
    
    with open(nucl_file, 'r') as handle:
        seq_records = SeqIO.parse(handle, 'fasta')
        for seq_record in seq_records:
            pool.apply_async(pla_frac_calcu, (seq_record, genus_name, pla_acc, que))
        
    pool.close()
    
    count = 0
    plasmidness_data = []
    with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            if not que.empty():
                value = que.get(True)
                plasmidness_data.append(value)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()
    plasmidness_data = pd.concat(plasmidness_data, ignore_index=True)
    plasmidness_data.to_csv(f'{target_dir}/IMGPR_plasmid_plasmidness.tsv', sep='\t', index=False)

Escherichia(24373): 100%|█████████████████████████████████████| 24.4k/24.4k [4:52:52<00:00, 1.39B/s]
Klebsiella(5897): 100%|███████████████████████████████████████| 5.90k/5.90k [1:45:52<00:00, 1.08s/B]
Staphylococcus(12910): 100%|██████████████████████████████████| 12.9k/12.9k [1:18:27<00:00, 2.74B/s]
Pseudomonas(4977): 100%|██████████████████████████████████████| 4.98k/4.98k [1:05:25<00:00, 1.27B/s]
Bacillus(5802): 100%|███████████████████████████████████████████| 5.80k/5.80k [36:35<00:00, 2.64B/s]
Salmonella(4575): 100%|█████████████████████████████████████████| 4.58k/4.58k [26:28<00:00, 2.88B/s]
Streptococcus(4736): 100%|██████████████████████████████████████| 4.74k/4.74k [18:03<00:00, 4.37B/s]
Streptomyces(2755): 100%|███████████████████████████████████████| 2.75k/2.75k [37:29<00:00, 1.22B/s]
Acinetobacter(4970): 100%|██████████████████████████████████████| 4.97k/4.97k [25:28<00:00, 3.25B/s]
Enterococcus(4633): 100%|███████████████████████████████████████| 4.63k/4.63k [25:55<00:00,